# MIP Guatemala 2013 — exploración reproducible

Este cuaderno es una vista de exploración de los CSV validados. La fuente de verdad computacional es `reproducir_mip_guatemala_2013.py`. No contiene escenarios de etanol ni actualizaciones a otros años.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / '02_resultados_y_diccionario').exists():
    candidate = ROOT.parent
    if (candidate / '02_resultados_y_diccionario').exists():
        ROOT = candidate
RESULTS = ROOT / '02_resultados_y_diccionario'
ROOT

## Metadatos y controles

In [ ]:
metadata = json.loads((RESULTS / 'metadatos_dataset.json').read_text(encoding='utf-8'))
controls = pd.read_csv(ROOT / '05_verificacion' / 'controles_reproduccion.csv')
display(pd.Series(metadata, name='valor').to_frame())
display(controls[['control_id', 'description', 'status', 'value', 'tolerance']])

## Carga de matrices

In [ ]:
def load_matrix(name):
    frame = pd.read_csv(RESULTS / 'matrices' / name)
    return frame[['codigo', 'producto']], frame.iloc[:, 2:].to_numpy(float)

products, Z_d = load_matrix('Z_domestica_2013.csv')
_, Z_m = load_matrix('Z_importada_2013.csv')
_, A_d = load_matrix('A_domestica_2013.csv')
_, L_d = load_matrix('Leontief_domestica_2013.csv')
{'Z_domestica': Z_d.shape, 'Z_importada': Z_m.shape, 'A_domestica': A_d.shape, 'Leontief': L_d.shape}

## Verificación matricial independiente

In [ ]:
residuo = (np.eye(152) - A_d) @ L_d - np.eye(152)
pd.Series({
    'suma_Z_domestica': Z_d.sum(),
    'suma_Z_importada': Z_m.sum(),
    'residuo_maximo_inversa': np.abs(residuo).max(),
})

## Producto energético agregado P068

La nomenclatura agrupa gasolinas, diésel oil y fuel oils. Cualquier desagregación posterior debe declarar una fuente y una regla propias.

In [ ]:
i = products.index[products['codigo'].eq('P068')][0]
pd.Series({
    'producto': products.loc[i, 'producto'],
    'ventas_intermedias_domesticas': Z_d[i, :].sum(),
    'compras_intermedias_domesticas': Z_d[:, i].sum(),
    'compras_intermedias_importadas': Z_m[:, i].sum(),
})